# Sistema Inteligente para Recomendación y Optimización de Portafolios de Concursos
**Filtro determinístico + Lógica difusa (compatibilidad) + Algoritmo genético con riesgo difuso en el fitness** sobre datos abiertos del OECE (SEACE)

Flujo implementado en este notebook:

`Ficha técnica (simulada) → Concursos OECE → Filtro duro → Lógica difusa (0-100) → Preselección → AG de portafolio (fitness con riesgo difuso como penalidad) → Resultados`

El módulo de IA generativa (chatbot) queda fuera del alcance: la ficha técnica se define a mano simulando la salida del chatbot, y la explicación final se genera con una plantilla.

## 1. Instalación e imports

In [ ]:
!pip install -q orjson

import gzip, json, re, os, unicodedata, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

try:
    import orjson
    loads = orjson.loads
except ImportError:
    loads = json.loads

pd.set_option("display.max_colwidth", 70)
SEED = 42
rng = np.random.default_rng(SEED)

## 2. Ficha técnica de la empresa (simulada)
En el sistema completo esta ficha la construye el chatbot. Aquí se fija a mano. Categorías según OCDS: `goods` = bienes, `services` = servicios, `works` = obras.

In [ ]:
FICHA = {
    "empresa": "TechSolutions SAC (ejemplo)",
    "rubro_keywords": [
        "software", "licencias", "computadora", "computadoras", "computo", "laptop", "laptops",
        "servidor", "servidores", "impresora", "impresoras", "informatica", "informaticos",
        "hardware", "red de datos", "cableado estructurado", "switch", "firewall",
        "tecnologia de la informacion", "soporte tecnico", "aplicativo", "plataforma web",
        "sistema informatico", "equipos de computo", "data center", "ciberseguridad",
    ],
    "categorias": ["goods", "services"],
    "capacidad_financiera": 1_500_000,     # monto máximo de contrato que puede ejecutar (S/)
    "monto_minimo_interes": 20_000,
    "presupuesto_postulaciones": 120_000,  # S/ para preparar y atender la cartera
    "experiencia_anios": 5,
    "personal_disponible": 6,
    "capacidad_operativa": 3,              # proyectos simultáneos (K máximo)
    "departamento_base": "LIMA",
    "cobertura": "nacional",               # "nacional" o "regional"
    "departamentos_cobertura": ["LIMA", "CALLAO", "ICA", "JUNIN"],
    "ventana_disponible": ("2026-10-01", "2026-12-31"),
    "certificaciones": ["RNP", "ISO 9001"],
}

## 3. Descarga de datos abiertos del OECE
Fuente: publicación OCDS del OECE en el Registro de Datos de Open Contracting Partnership (licencia CC BY 4.0), obtenida de `contratacionesabiertas.oece.gob.pe`. Se usa el archivo del año en curso en formato JSON Lines comprimido (~140 MB).

In [ ]:
ANIO = 2026
URL = f"https://data.open-contracting.org/es/publication/135/download?name={ANIO}.jsonl.gz"
ARCHIVO = f"oece_{ANIO}.jsonl.gz"

def descargar(url, destino):
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        print("Ya descargado:", destino)
        return
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total, hecho = int(r.headers.get("content-length", 0)), 0
        with open(destino, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                hecho += len(chunk)
                if total:
                    print(f"\r{hecho/1e6:.0f}/{total/1e6:.0f} MB", end="")
    print("\nListo:", destino)

descargar(URL, ARCHIVO)

In [ ]:
# Vistazo a la estructura de un proceso (útil si hay que ajustar campos)
with gzip.open(ARCHIVO, "rt", encoding="utf-8") as f:
    ejemplo = loads(f.readline())
ejemplo = ejemplo.get("compiledRelease", ejemplo)
print("Claves:", list(ejemplo.keys()))
print(json.dumps(ejemplo.get("tender", {}), ensure_ascii=False, indent=1)[:2000])

## 4. Extracción de campos relevantes

In [ ]:
def norm(txt):
    txt = unicodedata.normalize("NFKD", str(txt or "")).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", txt.lower()).strip()

def departamento(rel):
    buyer_id = (rel.get("buyer") or {}).get("id")
    for p in rel.get("parties") or []:
        if p.get("id") == buyer_id or "buyer" in (p.get("roles") or []):
            a = p.get("address") or {}
            for k in ("department", "region", "locality"):
                v = a.get(k)
                if isinstance(v, dict):
                    v = v.get("name") or v.get("description")
                if v:
                    return norm(v).upper()
    return "DESCONOCIDO"

def extraer(rel):
    t = rel.get("tender") or {}
    val = t.get("value") or {}
    tp = t.get("tenderPeriod") or {}
    cp = t.get("contractPeriod") or {}
    items = t.get("items") or []
    txt_items = " ".join(
        f"{i.get('description', '')} {(i.get('classification') or {}).get('description', '')}"
        for i in items[:30]
    )
    return {
        "ocid": rel.get("ocid"),
        "titulo": (t.get("title") or "")[:300],
        "descripcion": (t.get("description") or "")[:500],
        "texto_items": txt_items[:1500],
        "entidad": (rel.get("buyer") or {}).get("name", ""),
        "departamento": departamento(rel),
        "categoria": t.get("mainProcurementCategory"),
        "metodo": t.get("procurementMethodDetails") or t.get("procurementMethod"),
        "estado": t.get("status"),
        "monto": val.get("amount"),
        "moneda": val.get("currency"),
        "inicio_postulacion": tp.get("startDate"),
        "fin_postulacion": tp.get("endDate"),
        "duracion_dias": cp.get("durationInDays"),
        "fecha_publicacion": t.get("datePublished") or rel.get("date"),
    }

filas = []
with gzip.open(ARCHIVO, "rt", encoding="utf-8") as f:
    for n, linea in enumerate(f, 1):
        try:
            obj = loads(linea)
        except Exception:
            continue
        filas.append(extraer(obj.get("compiledRelease", obj)))
        if n % 50_000 == 0:
            print(n, "procesos leídos")

df_raw = pd.DataFrame(filas)
del filas
for c in ["inicio_postulacion", "fin_postulacion", "fecha_publicacion"]:
    df_raw[c] = pd.to_datetime(df_raw[c], errors="coerce", utc=True).dt.tz_convert(None)
df_raw["monto"] = pd.to_numeric(df_raw["monto"], errors="coerce")
df_raw["duracion_dias"] = pd.to_numeric(df_raw["duracion_dias"], errors="coerce")
df_raw.to_pickle(f"oece_{ANIO}_procesos.pkl")
print(df_raw.shape)

In [ ]:
print(df_raw["categoria"].value_counts(dropna=False), "\n")
print(df_raw["estado"].value_counts(dropna=False), "\n")
print(df_raw["metodo"].value_counts().head(10), "\n")
print(df_raw["departamento"].value_counts().head(10))

## 5. Concursos vigentes
Un concurso es vigente si su periodo de postulación sigue abierto en la fecha de referencia. Como el OECE publica cortes mensuales, si hay pocos procesos abiertos en la fecha de corte se retrocede la fecha de referencia de 7 en 7 días (simulando que se consulta en ese momento).

In [ ]:
MIN_VIGENTES = 50
FECHA_CORTE = df_raw["fecha_publicacion"].max().normalize()
FECHA_BASE = min(pd.Timestamp.today().normalize(), FECHA_CORTE)

def abiertos(f):
    ini_ok = df_raw["inicio_postulacion"].isna() | (df_raw["inicio_postulacion"] <= f)
    return df_raw[(df_raw["fin_postulacion"] >= f) & ini_ok]

retro = 0
vigentes = abiertos(FECHA_BASE)
while len(vigentes) < MIN_VIGENTES and retro < 180:
    retro += 7
    vigentes = abiertos(FECHA_BASE - pd.Timedelta(days=retro))
FECHA_REF = FECHA_BASE - pd.Timedelta(days=retro)
vigentes = vigentes.copy()

print("Fecha de corte del dataset:", FECHA_CORTE.date())
print("Fecha de referencia usada :", FECHA_REF.date(), f"(retroceso {retro} días)")
print("Concursos vigentes:", len(vigentes))

## 6. Filtro determinístico
Elimina concursos que no admiten gradualidad: rubro ajeno, categoría no atendida, modalidad a la que no se puede postular, monto fuera de la capacidad financiera, cobertura geográfica no atendible y estados inválidos.

Nota: los datos abiertos no traen los requisitos de las bases (certificaciones, experiencia mínima obligatoria). Si se agregan como columnas `certificacion_requerida` o `experiencia_min_requerida`, el filtro las aplica.

In [ ]:
TIPO_CAMBIO_USD = 3.75
PATRONES = [re.compile(r"\b" + re.escape(norm(k)) + r"\b") for k in FICHA["rubro_keywords"]]

def afinidad(texto):
    t = norm(texto)
    hits = sum(1 for p in PATRONES if p.search(t))
    return min(1.0, hits / 3)  # 3 o más coincidencias = afinidad máxima

df = vigentes.copy()
df["monto_pen"] = np.where(df["moneda"] == "USD", df["monto"] * TIPO_CAMBIO_USD,
                           np.where(df["moneda"].isna() | (df["moneda"] == "PEN"), df["monto"], np.nan))
df["afinidad"] = (df["titulo"] + " " + df["descripcion"] + " " + df["texto_items"]).map(afinidad)
metodo_n = df["metodo"].map(norm)

reglas_filtro = {
    "Estado inválido": ~df["estado"].isin(["cancelled", "unsuccessful", "withdrawn"]),
    "Categoría no atendida": df["categoria"].isin(FICHA["categorias"]),
    "Contratación directa": ~metodo_n.str.contains("contratacion directa", na=False),
    "Monto no válido": df["monto_pen"].notna() & (df["monto_pen"] > 0),
    "Monto supera capacidad": df["monto_pen"] <= FICHA["capacidad_financiera"],
    "Monto menor al de interés": df["monto_pen"] >= FICHA["monto_minimo_interes"],
    "Rubro ajeno": df["afinidad"] > 0,
}
if FICHA["cobertura"] != "nacional":
    reglas_filtro["Fuera de cobertura"] = df["departamento"].isin(FICHA["departamentos_cobertura"] + ["DESCONOCIDO"])
if "certificacion_requerida" in df:
    reglas_filtro["Certificación ausente"] = df["certificacion_requerida"].isna() | df["certificacion_requerida"].isin(FICHA["certificaciones"])
if "experiencia_min_requerida" in df:
    reglas_filtro["Experiencia insuficiente"] = df["experiencia_min_requerida"].fillna(0) <= FICHA["experiencia_anios"]

mascara = pd.Series(True, index=df.index)
resumen = []
for nombre, regla in reglas_filtro.items():
    antes = mascara.sum()
    mascara &= regla.fillna(False)
    resumen.append((nombre, antes - mascara.sum()))

factibles = df[mascara].copy()
print(pd.DataFrame(resumen, columns=["Regla", "Descartados"]).to_string(index=False))
print("\nConcursos factibles:", len(factibles))

## 7. Variables de entrada para la lógica difusa
Como los datos abiertos no traen todo lo necesario, algunas variables se estiman con supuestos explícitos (se pueden ajustar):

| Variable | Cálculo |
|---|---|
| `ratio_monto` | monto del concurso / capacidad financiera |
| `experiencia` | años de la empresa / años requeridos estimados (1 + monto/300 000) |
| `plazo` | días entre la fecha de referencia y el cierre de postulación |
| `tecnica` | afinidad textual con el rubro (0-1) |
| `personal` | personal disponible / personal estimado del concurso |
| `ubicacion` | 1 mismo departamento, 0.7 en cobertura, 0.5 desconocido, 0.3 otro |

In [ ]:
def estimar_personal(monto, cat):
    base = 1 if cat == "goods" else 2
    return int(np.clip(base + monto // 250_000, 1, 8))

def estimar_duracion(monto, cat, dur):
    if pd.notna(dur) and dur > 0:
        return int(dur)
    lo, hi = (30, 90) if cat == "goods" else (60, 180)
    return int(np.clip(lo + monto / 10_000, lo, hi))

def puntaje_ubicacion(dep):
    if dep == FICHA["departamento_base"]: return 1.0
    if dep in FICHA["departamentos_cobertura"]: return 0.7
    if dep == "DESCONOCIDO": return 0.5
    return 0.3

fac = factibles
fac["personal_req"] = [estimar_personal(m, c) for m, c in zip(fac["monto_pen"], fac["categoria"])]
fac["duracion_est"] = [estimar_duracion(m, c, d) for m, c, d in zip(fac["monto_pen"], fac["categoria"], fac["duracion_dias"])]
fac["exp_req"] = 1 + fac["monto_pen"] / 300_000

fac["ratio_monto"] = (fac["monto_pen"] / FICHA["capacidad_financiera"]).clip(0, 1)
fac["experiencia"] = (FICHA["experiencia_anios"] / fac["exp_req"]).clip(0, 2)
fac["plazo"] = (fac["fin_postulacion"] - FECHA_REF).dt.days.clip(0, 30)
fac["tecnica"] = fac["afinidad"]
fac["personal"] = (FICHA["personal_disponible"] / fac["personal_req"]).clip(0, 3)
fac["ubicacion"] = fac["departamento"].map(puntaje_ubicacion)

ENTRADAS = ["ratio_monto", "experiencia", "plazo", "tecnica", "personal", "ubicacion"]
fac[ENTRADAS].describe().round(2)

## 8. Motor de lógica difusa (Mamdani)
Funciones de pertenencia triangulares (con hombros en los extremos), inferencia min-max y defuzzificación por centroide. Salida: compatibilidad 0-100.

In [ ]:
def trimf(x, a, b, c):
    x = np.asarray(x, dtype=float)
    izq = (x - a) / (b - a) if b > a else np.ones_like(x)
    der = (c - x) / (c - b) if c > b else np.ones_like(x)
    return np.clip(np.minimum(izq, der), 0, 1)

VARS = {
    "ratio_monto": {"pequeno": (0, 0, 0.25), "adecuado": (0.1, 0.45, 0.8), "exigente": (0.6, 1, 1)},
    "experiencia": {"insuficiente": (0, 0, 0.8), "aceptable": (0.5, 1, 1.5), "fuerte": (1.2, 2, 2)},
    "plazo":       {"ajustado": (0, 0, 7), "razonable": (4, 10, 18), "holgado": (14, 30, 30)},
    "tecnica":     {"baja": (0, 0, 0.4), "media": (0.2, 0.5, 0.8), "alta": (0.6, 1, 1)},
    "personal":    {"limitada": (0, 0, 1), "suficiente": (0.7, 1.2, 2), "amplia": (1.6, 3, 3)},
    "ubicacion":   {"baja": (0, 0, 0.5), "media": (0.3, 0.55, 0.8), "alta": (0.6, 1, 1)},
}
Y = np.linspace(0, 100, 501)
SALIDA = {"baja": (0, 0, 40), "media": (25, 50, 70), "alta": (55, 75, 90), "muy_alta": (80, 100, 100)}
MF_SALIDA = {k: trimf(Y, *p) for k, p in SALIDA.items()}

REGLAS = [
    ({"tecnica": "alta", "experiencia": "fuerte", "ratio_monto": "adecuado"}, "muy_alta"),
    ({"tecnica": "alta", "ubicacion": "alta", "plazo": "holgado"}, "muy_alta"),
    ({"tecnica": "alta", "experiencia": "aceptable", "personal": "suficiente"}, "alta"),
    ({"tecnica": "alta", "plazo": "razonable"}, "alta"),
    ({"tecnica": "alta", "ratio_monto": "pequeno"}, "alta"),
    ({"tecnica": "media", "ubicacion": "alta", "personal": "amplia"}, "alta"),
    ({"tecnica": "media", "experiencia": "fuerte", "personal": "amplia"}, "alta"),
    ({"tecnica": "media", "experiencia": "aceptable"}, "media"),
    ({"ratio_monto": "exigente", "experiencia": "fuerte", "personal": "amplia"}, "alta"),
    ({"plazo": "ajustado", "tecnica": "alta", "personal": "amplia"}, "media"),
    ({"ubicacion": "baja", "tecnica": "alta"}, "media"),
    ({"tecnica": "baja"}, "baja"),
    ({"experiencia": "insuficiente"}, "baja"),
    ({"ratio_monto": "exigente", "personal": "limitada"}, "baja"),
    ({"plazo": "ajustado", "personal": "limitada"}, "baja"),
    ({"ubicacion": "baja", "ratio_monto": "exigente"}, "baja"),
]

def evaluar_difuso(entrada):
    grados = {v: {t: float(trimf(entrada[v], *p)) for t, p in terms.items()} for v, terms in VARS.items()}
    agregada = np.zeros_like(Y)
    mejor_regla, mejor_w = None, 0.0
    for ant, cons in REGLAS:
        w = min(grados[v][t] for v, t in ant.items())
        if w > 0:
            agregada = np.maximum(agregada, np.minimum(w, MF_SALIDA[cons]))
            if w > mejor_w:
                mejor_regla, mejor_w = (ant, cons), w
    if agregada.sum() == 0:
        return 0.0, None
    return float((Y * agregada).sum() / agregada.sum()), mejor_regla

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
universos = {"ratio_monto": (0, 1), "experiencia": (0, 2), "plazo": (0, 30),
             "tecnica": (0, 1), "personal": (0, 3), "ubicacion": (0, 1)}
for ax, (v, (lo, hi)) in zip(axes.flat, universos.items()):
    x = np.linspace(lo, hi, 300)
    for t, p in VARS[v].items():
        ax.plot(x, trimf(x, *p), label=t)
    ax.set_title(v); ax.legend(fontsize=8)
for t, mf in MF_SALIDA.items():
    axes.flat[6].plot(Y, mf, label=t)
axes.flat[6].set_title("compatibilidad (salida)"); axes.flat[6].legend(fontsize=8)
axes.flat[7].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
def lectura(p):
    if p >= 85: return "Muy alta compatibilidad"
    if p >= 75: return "Alta compatibilidad"
    if p >= 60: return "Buena compatibilidad"
    if p >= 40: return "Compatibilidad media"
    return "Baja compatibilidad"

res = [evaluar_difuso(r) for r in fac[ENTRADAS].to_dict("records")]
fac["score"] = [round(p, 1) for p, _ in res]
fac["regla_dominante"] = [f"{r[0]} -> {r[1]}" if r else "" for _, r in res]
fac["lectura"] = fac["score"].map(lectura)

top_individual = fac.sort_values("score", ascending=False)
top_individual[["titulo", "entidad", "departamento", "monto_pen", "score", "lectura"]].head(15)

## 9. Preselección de candidatos para el AG
Se combinan umbral mínimo y top-N para reducir el espacio de búsqueda.

In [ ]:
UMBRAL = 60
TOP_N = 20

cand = top_individual[top_individual["score"] >= UMBRAL].head(TOP_N)
if len(cand) < 5:
    print(f"Pocos concursos superan {UMBRAL}; se toman los {TOP_N} mejores sin umbral.")
    cand = top_individual.head(TOP_N)
cand = cand.reset_index(drop=True)
cand.index = [f"C{i+1}" for i in range(len(cand))]
N = len(cand)
print("Candidatos (N):", N)
cand[["titulo", "monto_pen", "score", "personal_req", "duracion_est"]]

## 10. Evaluador de recursos y calendario
Supuestos:
- **Costo de atender** = costo fijo de preparación + % del monto (garantías, capital de trabajo).
- **Periodo de preparación** = desde la fecha de referencia hasta el cierre de postulación.
- **Periodo de ejecución** = inicia N días después del cierre y dura `duracion_est`.
- **conflicto(i,j)** = 0.3 · solape de preparación + 0.7 · solape de ejecución (cada uno normalizado por el periodo más corto), en [0, 1].
- **fuera_ventana(i)** = fracción de la ejecución fuera de la ventana disponible de la empresa.

In [ ]:
COSTO_FIJO = 2_500
PCT_COSTO = 0.06
DIAS_HASTA_EJECUCION = 20

cand["costo"] = COSTO_FIJO + PCT_COSTO * cand["monto_pen"]
prep_ini = np.full(N, FECHA_REF.value)
prep_fin = cand["fin_postulacion"].values.astype("datetime64[ns]").astype(np.int64)
ejec_ini = prep_fin + np.int64(DIAS_HASTA_EJECUCION * 86400e9)
ejec_fin = ejec_ini + (cand["duracion_est"].values * 86400e9).astype(np.int64)

def solape_norm(ini, fin):
    inter = np.maximum(0, np.minimum(fin[:, None], fin[None, :]) - np.maximum(ini[:, None], ini[None, :]))
    largo = np.maximum(1, np.minimum((fin - ini)[:, None], (fin - ini)[None, :]))
    return inter / largo

CONFLICTO = 0.3 * solape_norm(prep_ini, prep_fin) + 0.7 * solape_norm(ejec_ini, ejec_fin)
np.fill_diagonal(CONFLICTO, 0)
CONF_SUP = np.triu(CONFLICTO, 1)

v_ini, v_fin = [pd.Timestamp(d).value for d in FICHA["ventana_disponible"]]
dentro = np.maximum(0, np.minimum(ejec_fin, v_fin) - np.maximum(ejec_ini, v_ini))
FUERA = 1 - dentro / np.maximum(1, ejec_fin - ejec_ini)

S = cand["score"].values.astype(float)
COSTO = cand["costo"].values.astype(float)
PERS = cand["personal_req"].values.astype(float)

pd.DataFrame(CONFLICTO, index=cand.index, columns=cand.index).round(2)

## 10.1 Riesgo difuso del portafolio (segundo sistema difuso)
El primer sistema difuso puntúa **cada concurso** (compatibilidad 0-100). Este segundo sistema puntúa **el portafolio completo** (riesgo 0-100) y se usa como **penalidad** en el fitness: a mayor riesgo, menor fitness.

| Entrada | Qué mide |
|---|---|
| `cantidad` | concursos elegidos / K máximo |
| `capital` | monto total comprometido / capacidad financiera |
| `personal` | personal requerido total / personal disponible |
| `fechas` | conflicto promedio entre pares de concursos elegidos |

Por eso el riesgo no se puede calcular concurso por concurso: depende de la **combinación** (dos concursos que solos son seguros pueden ser riesgosos juntos). Las reglas capturan ideas como *capital excedido → riesgo crítico* o *muchos concursos aunque haya holgura → riesgo medio*.


In [ ]:
# ---- Riesgo difuso de PORTAFOLIO (Mamdani) ----
# Entradas (miden qué tan "cargada" queda la empresa con el portafolio x):
#   cantidad : nº de concursos elegidos / K_MAX
#   capital  : monto total comprometido / capacidad financiera        (0-2)
#   personal : personal total requerido / personal disponible          (0-2)
#   fechas   : conflicto promedio entre pares de concursos elegidos    (0-1)
# Salida: riesgo 0-100 (100 = riesgo crítico). En el fitness se RESTA.

MONTO = cand["monto_pen"].values.astype(float)
CAP_FIN = FICHA["capital_disponible"] if "capital_disponible" in FICHA else FICHA["capacidad_financiera"]
K_TOPE = FICHA["capacidad_operativa"]
PERS_DISP = FICHA["personal_disponible"]

VARS_R = {
    "cantidad": {"baja": (0, 0, 0.5), "media": (0.3, 0.6, 0.85), "alta": (0.6, 1, 1)},
    "capital":  {"holgado": (0, 0, 0.6), "ajustado": (0.4, 0.8, 1.2), "excedido": (1, 2, 2)},
    "personal": {"holgado": (0, 0, 0.7), "justo": (0.5, 0.9, 1.3), "excedido": (1, 2, 2)},
    "fechas":   {"bajo": (0, 0, 0.3), "medio": (0.15, 0.4, 0.7), "alto": (0.5, 1, 1)},
}
YR = np.linspace(0, 100, 201)
SALIDA_R = {"bajo": (0, 0, 30), "medio": (15, 40, 65), "alto": (50, 70, 90), "critico": (80, 100, 100)}
MF_R = {k: trimf(YR, *p) for k, p in SALIDA_R.items()}

REGLAS_R = [
    ({"capital": "excedido"}, "critico"),
    ({"personal": "excedido"}, "alto"),
    ({"cantidad": "alta", "fechas": "alto"}, "critico"),
    ({"capital": "ajustado", "cantidad": "alta"}, "alto"),
    ({"personal": "justo", "cantidad": "alta"}, "alto"),
    ({"capital": "ajustado", "personal": "justo"}, "alto"),
    ({"fechas": "alto"}, "alto"),
    ({"capital": "ajustado", "cantidad": "media"}, "medio"),
    ({"personal": "justo", "cantidad": "media"}, "medio"),
    ({"fechas": "medio", "cantidad": "alta"}, "medio"),
    ({"cantidad": "alta", "capital": "holgado", "personal": "holgado"}, "medio"),
    ({"capital": "holgado", "personal": "holgado", "fechas": "bajo", "cantidad": "media"}, "bajo"),
    ({"cantidad": "baja"}, "bajo"),
    ({"capital": "holgado", "cantidad": "media"}, "bajo"),
    ({"personal": "holgado", "fechas": "bajo"}, "bajo"),
]

def entradas_riesgo(x):
    k = float(x.sum())
    pares = k * (k - 1) / 2
    return {
        "cantidad": min(1.0, k / K_TOPE),
        "capital": min(2.0, float(MONTO @ x) / CAP_FIN),
        "personal": min(2.0, float(PERS @ x) / PERS_DISP),
        "fechas": min(1.0, float(x @ CONF_SUP @ x) / pares) if pares > 0 else 0.0,
    }

def riesgo_difuso(x):
    """Devuelve (riesgo 0-100, entradas usadas)."""
    if x.sum() == 0:
        return 0.0, entradas_riesgo(x)
    ent = entradas_riesgo(x)
    grados = {v: {t: float(trimf(ent[v], *p)) for t, p in terms.items()} for v, terms in VARS_R.items()}
    agregada = np.zeros_like(YR)
    for ant, cons in REGLAS_R:
        w = min(grados[v][t] for v, t in ant.items())
        if w > 0:
            agregada = np.maximum(agregada, np.minimum(w, MF_R[cons]))
    if agregada.sum() == 0:
        return 0.0, ent
    return float((YR * agregada).sum() / agregada.sum()), ent


## 11. Algoritmo genético del portafolio
- **Gen**: xᵢ ∈ {0,1}, incluir o no el concurso Cᵢ.
- **Cromosoma**: cadena binaria de longitud N (un portafolio completo).
- **Fitness**: `Σ xᵢ·sᵢ − λ₁·P_recursos − λ₂·P_fechas − λ₃·P_presupuesto − λ₄·P_ventana − λ₅·Riesgo_difuso(x)`
- **Restricciones duras** (vía reparación): `K_MIN ≤ Σxᵢ ≤ K_MAX` y, si `PRESUPUESTO_DURO`, costo total ≤ presupuesto.
- **Operadores**: selección por torneo, cruce uniforme, mutación bit a bit, elitismo y reparación.
- **Riesgo_difuso(x)** (sección 10.1) es un número 0-100 que se **resta**: si el portafolio es más riesgoso, el fitness baja.
- Nota: `P_recursos` y `P_fechas` ya castigan parte de lo mismo que el riesgo. Si el riesgo domina demasiado, baja esos λ o `LAMBDA['riesgo']` para evitar penalizar dos veces.

In [ ]:
K_MIN, K_MAX = 2, FICHA["capacidad_operativa"]
PRESUP = FICHA["presupuesto_postulaciones"]
PERSONAL = FICHA["personal_disponible"]
PRESUPUESTO_DURO = True
LAMBDA = {"recursos": 1.0, "fechas": 0.6, "presupuesto": 5.0, "ventana": 0.2, "riesgo": 0.8}

AG = {"poblacion": 80, "generaciones": 200, "p_cruce": 0.9, "p_mut": 1 / max(N, 1),
      "torneo": 3, "elite": 2, "paciencia": 40}

def penalizaciones(x):
    return {
        "recursos": max(0.0, PERS @ x - PERSONAL) / PERSONAL * 100,
        "fechas": float(x @ CONF_SUP @ x) * 100,
        "presupuesto": max(0.0, COSTO @ x - PRESUP) / PRESUP * 100,
        "ventana": float(FUERA @ x) * 100,
        "riesgo": riesgo_difuso(x)[0],
    }

def fitness(x):
    p = penalizaciones(x)
    return float(S @ x) - sum(LAMBDA[k] * v for k, v in p.items())

def reparar(x):
    x = x.copy()
    eficiencia = S / COSTO
    while x.sum() > K_MAX or (PRESUPUESTO_DURO and COSTO @ x > PRESUP):
        activos = np.flatnonzero(x)
        x[activos[np.argmin(eficiencia[activos])]] = 0
    for i in np.argsort(-S):
        if x.sum() >= K_MIN:
            break
        if x[i] == 0 and (not PRESUPUESTO_DURO or COSTO @ x + COSTO[i] <= PRESUP):
            x[i] = 1
    return x

def torneo(pob, fit):
    idx = rng.choice(len(pob), AG["torneo"], replace=False)
    return pob[idx[np.argmax(fit[idx])]]

def algoritmo_genetico():
    pob = (rng.random((AG["poblacion"], N)) < K_MAX / N).astype(int)
    pob = np.array([reparar(x) for x in pob])
    fit = np.array([fitness(x) for x in pob])
    hist_mejor, hist_media = [], []
    mejor_global, sin_mejora = -np.inf, 0

    for gen in range(AG["generaciones"]):
        orden = np.argsort(-fit)
        nueva = [pob[i].copy() for i in orden[:AG["elite"]]]
        while len(nueva) < AG["poblacion"]:
            p1, p2 = torneo(pob, fit), torneo(pob, fit)
            if rng.random() < AG["p_cruce"]:
                m = rng.random(N) < 0.5
                h1, h2 = np.where(m, p1, p2), np.where(m, p2, p1)
            else:
                h1, h2 = p1.copy(), p2.copy()
            for h in (h1, h2):
                flip = rng.random(N) < AG["p_mut"]
                h[flip] = 1 - h[flip]
                nueva.append(reparar(h))
        pob = np.array(nueva[:AG["poblacion"]])
        fit = np.array([fitness(x) for x in pob])

        hist_mejor.append(fit.max()); hist_media.append(fit.mean())
        if fit.max() > mejor_global + 1e-9:
            mejor_global, sin_mejora = fit.max(), 0
        else:
            sin_mejora += 1
            if sin_mejora >= AG["paciencia"]:
                break

    unicos = {}
    for x, fx in zip(pob, fit):
        unicos.setdefault(tuple(x), fx)
    ranking = sorted(unicos.items(), key=lambda kv: -kv[1])
    return ranking, hist_mejor, hist_media, gen + 1

ranking, hist_mejor, hist_media, gens = algoritmo_genetico()
mejor_x = np.array(ranking[0][0])
print(f"Generaciones ejecutadas: {gens}")
print("Mejor cromosoma:", ";".join(map(str, mejor_x)), f"| fitness = {ranking[0][1]:.2f}")

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(hist_mejor, label="Mejor fitness")
plt.plot(hist_media, label="Fitness promedio", alpha=0.7)
plt.xlabel("Generación"); plt.ylabel("Fitness"); plt.title("Convergencia del AG")
plt.legend(); plt.grid(alpha=0.3); plt.show()

### 11.1 Validación contra búsqueda exhaustiva
Con K pequeño se pueden enumerar todas las combinaciones factibles y verificar si el AG encontró el óptimo. Con N o K grandes esto deja de ser viable y ahí se justifica el AG.

In [ ]:
mejor_bf, x_bf, evaluados = -np.inf, None, 0
for k in range(K_MIN, K_MAX + 1):
    for comb in itertools.combinations(range(N), k):
        x = np.zeros(N, dtype=int); x[list(comb)] = 1
        if PRESUPUESTO_DURO and COSTO @ x > PRESUP:
            continue
        evaluados += 1
        fx = fitness(x)
        if fx > mejor_bf:
            mejor_bf, x_bf = fx, x

print(f"Combinaciones factibles evaluadas: {evaluados}")
print(f"Óptimo exhaustivo: {list(cand.index[x_bf == 1])} | fitness = {mejor_bf:.2f}")
print(f"Mejor del AG     : {list(cand.index[mejor_x == 1])} | fitness = {ranking[0][1]:.2f}")
print("El AG encontró el óptimo" if abs(mejor_bf - ranking[0][1]) < 1e-6 else "El AG quedó cerca del óptimo")

## 12. Resultados para el usuario

In [ ]:
def resumen_portafolio(x):
    sel = cand[x == 1]
    p = penalizaciones(x)
    return {
        "concursos": " + ".join(sel.index),
        "suma_difusa": round(float(S @ x), 1),
        "costo_total": round(float(COSTO @ x)),
        "personal_req": int(PERS @ x),
        "conflicto_fechas": round(p["fechas"] / 100, 2),
        "riesgo_difuso": round(p["riesgo"], 1),
        "fitness": round(fitness(x), 2),
    }

alternativas = pd.DataFrame([resumen_portafolio(np.array(x)) for x, _ in ranking[:5]])
alternativas.index = [f"Opción {i+1}" for i in range(len(alternativas))]
alternativas

In [ ]:
def explicacion(x, top=5):
    # Plantilla que simula la respuesta del chatbot
    lineas = ["Top individual (lógica difusa):"]
    for i, (_, r) in enumerate(top_individual.head(top).iterrows(), 1):
        lineas.append(f"  {i}. {r['titulo'][:70]} | {r['entidad'][:40]} | S/ {r['monto_pen']:,.0f} | {r['score']:.0f}/100")
    sel = cand[x == 1]
    p = penalizaciones(x)
    lineas.append("\nPortafolio sugerido (algoritmo genético):")
    for cid, r in sel.iterrows():
        lineas.append(f"  {cid}: {r['titulo'][:70]} ({r['score']:.0f}/100, S/ {r['monto_pen']:,.0f})")
    lineas.append(
        f"\nMotivo: compatibilidad acumulada de {S @ x:.0f} puntos, costo de atención "
        f"S/ {COSTO @ x:,.0f} de S/ {PRESUP:,.0f} disponibles, "
        f"{int(PERS @ x)} de {PERSONAL} personas requeridas y "
        f"{'bajo' if p['fechas'] < 30 else 'moderado' if p['fechas'] < 70 else 'alto'} conflicto de fechas."
    )
    nivel = "bajo" if p["riesgo"] < 30 else "medio" if p["riesgo"] < 55 else "alto" if p["riesgo"] < 75 else "crítico"
    _, ent = riesgo_difuso(x)
    lineas.append(
        f"Riesgo difuso del portafolio: {p['riesgo']:.0f}/100 ({nivel}) | capital comprometido {ent['capital']:.0%} de la capacidad, "
        f"personal {ent['personal']:.0%} del disponible, {int(x.sum())} de {K_MAX} concursos posibles."
    )
    if p["recursos"] > 0:
        lineas.append("Advertencia: la cartera excede el personal disponible; considerar contratar apoyo temporal.")
    lineas.append("Limitación: costos, personal y duración son estimaciones; revisar las bases de cada concurso.")
    return "\n".join(lineas)

print(explicacion(mejor_x))

## 13. Parámetros a ajustar (sección 19 del documento)
- Preselección: `UMBRAL`, `TOP_N`
- Recursos: `COSTO_FIJO`, `PCT_COSTO`, `estimar_personal`, `estimar_duracion`
- Fechas: `DIAS_HASTA_EJECUCION`, pesos 0.3 / 0.7 del conflicto, `ventana_disponible`
- Tamaño del portafolio: `K_MIN`, `K_MAX`
- Penalizaciones: `LAMBDA` (incluye `LAMBDA['riesgo']`)
- Riesgo difuso: `VARS_R`, `SALIDA_R`, `REGLAS_R` (sección 10.1)
- AG: `AG` (población, generaciones, cruce, mutación, torneo, elitismo, paciencia)
- Salida: se devuelven las 5 mejores alternativas